# DeepONet vs SpectralSVR

Reads the committed comparison record, `benchmarks/operator_comparison.csv`, and fits nothing.
Regenerate the record with `just bench`; `scripts/run_benchmark.py` states what the table
establishes and what it does not.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from SpectralSVR.operators import summarise

repository = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / 'benchmarks').is_dir()
)
runs = pd.read_csv(repository / 'benchmarks' / 'operator_comparison.csv')

## Provenance

One row per dataset, operator and seed; every row says what produced it.

In [ ]:
provenance = ['revision', 'tree_clean', 'torch_version', 'deepxde_version', 'coeff_dtype', 'seed_set', 'quick']
assert not runs['quick'].any()
runs[provenance].drop_duplicates()

In [ ]:
runs[['dataset', 'operator', 'protocol', 'fit_covers', 'data_device', 'fit_device']].drop_duplicates()

## Centre and spread over the seed set

`mean_per_sample_relative_l2` is the convention neural-operator papers report; `rrse` is the
global form the library computes over every element at once.

In [ ]:
summary = summarise(runs)
columns = ['n_seeds', 'r2_mean', 'r2_std', 'mean_per_sample_relative_l2_mean',
           'mean_per_sample_relative_l2_std', 'rrse_mean', 'rrse_std', 'fit_seconds_mean']
summary[columns].round(4)

In [ ]:
relative_l2 = summary['mean_per_sample_relative_l2_mean'].unstack('operator')
spread = summary['mean_per_sample_relative_l2_std'].unstack('operator')
axes = relative_l2.plot.barh(xerr=spread, figsize=(9, 5), logx=True)
axes.set_xlabel('mean per-sample relative L2 (mean ± sd over seeds)')
plt.tight_layout()